In [9]:
# ============================================================
# CELL 1
# INSTALL LIBRARIES + IMPORTS + LOCAL EMBEDDING MODEL
# ============================================================

!pip install -q sentence-transformers faiss-cpu numpy


from google.colab import files
from sentence_transformers import SentenceTransformer

import numpy as np
import faiss
import pickle
import re
import os


# ============================================================
# Load local embedding model
# ============================================================

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(
    EMBEDDING_MODEL
)


print("Local embedding model loaded successfully.")

print(
    "Model:",
    EMBEDDING_MODEL
)

print(
    "No API key is required."
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local embedding model loaded successfully.
Model: sentence-transformers/all-MiniLM-L6-v2
No API key is required.


In [11]:
# ============================================================
# CELL 2
# PHASE 1
#
# Upload 3 TXT files
# Dynamic Chunking
# Create Embeddings
# Store in FAISS
# Save FAISS + Metadata
#
# RUN THIS PHASE ONLY ONCE
# ============================================================


print("=" * 70)
print("PHASE 1 - BUILD VECTOR DATABASE")
print("=" * 70)


# ============================================================
# 1. Upload exactly 3 TXT files
# ============================================================

print(
    "\nUpload exactly 3 TXT files "
    "from different fields."
)

print("\nExample:")
print("- football.txt")
print("- saudi_food.txt")
print("- artificial_intelligence.txt")


uploaded = files.upload()


txt_files = [

    filename

    for filename in uploaded.keys()

    if filename.lower().endswith(".txt")

]


if len(txt_files) != 3:

    raise ValueError(

        f"You must upload exactly 3 TXT files. "
        f"You uploaded {len(txt_files)}."

    )


print("\nUploaded files:")


for filename in txt_files:

    print("-", filename)


# ============================================================
# 2. Read the three documents
# ============================================================

documents = []


for filename in txt_files:

    with open(

        filename,

        "r",

        encoding="utf-8"

    ) as file:

        text = file.read()


    documents.append({

        "filename":
            filename,

        "text":
            text

    })


    print(
        f"\nLoaded: {filename}"
    )

    print(
        f"Characters: {len(text)}"
    )


# ============================================================
# 3. Dynamic Text Chunking
# ============================================================

def dynamic_chunk_text(
    text,
    max_chunk_size=700
):

    """
    Dynamic Text Chunking

    1. Split text into paragraphs.
    2. Keep paragraphs together when possible.
    3. Combine small paragraphs.
    4. Split large paragraphs using sentences.
    """


    # Split text by blank lines
    paragraphs = re.split(

        r'\n\s*\n',

        text

    )


    # Remove empty paragraphs
    paragraphs = [

        paragraph.strip()

        for paragraph in paragraphs

        if paragraph.strip()

    ]


    chunks = []

    current_chunk = ""


    for paragraph in paragraphs:


        # ----------------------------------------------------
        # Case 1:
        # Paragraph fits inside current chunk
        # ----------------------------------------------------

        if (

            len(current_chunk)

            + len(paragraph)

            + 2

            <= max_chunk_size

        ):


            if current_chunk:

                current_chunk += (
                    "\n\n" + paragraph
                )

            else:

                current_chunk = paragraph


        # ----------------------------------------------------
        # Case 2:
        # Current chunk becomes too large
        # ----------------------------------------------------

        else:


            if current_chunk.strip():

                chunks.append(
                    current_chunk.strip()
                )

                current_chunk = ""


            # ------------------------------------------------
            # If the paragraph itself is very large,
            # split it by sentences
            # ------------------------------------------------

            if len(paragraph) > max_chunk_size:


                sentences = re.split(

                    r'(?<=[.!?])\s+',

                    paragraph

                )


                sentence_chunk = ""


                for sentence in sentences:


                    if (

                        len(sentence_chunk)

                        + len(sentence)

                        + 1

                        <= max_chunk_size

                    ):


                        if sentence_chunk:

                            sentence_chunk += (
                                " " + sentence
                            )

                        else:

                            sentence_chunk = sentence


                    else:


                        if sentence_chunk.strip():

                            chunks.append(
                                sentence_chunk.strip()
                            )


                        sentence_chunk = sentence


                if sentence_chunk.strip():

                    current_chunk = (
                        sentence_chunk.strip()
                    )


            else:

                current_chunk = paragraph


    # Add the final chunk
    if current_chunk.strip():

        chunks.append(
            current_chunk.strip()
        )


    return chunks


# ============================================================
# 4. Chunk all three documents
# ============================================================

all_chunks = []


for document in documents:


    filename = document["filename"]

    text = document["text"]


    chunks = dynamic_chunk_text(

        text,

        max_chunk_size=700

    )


    print(
        f"\n{filename}: "
        f"{len(chunks)} chunks"
    )


    for i, chunk in enumerate(chunks):


        all_chunks.append({

            "text":
                chunk,

            "source":
                filename,

            "chunk_number":
                i + 1

        })


print(
    "\nTotal chunks:",
    len(all_chunks)
)


# ============================================================
# 5. Display sample chunks
# ============================================================

print("\n")
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)


for i, item in enumerate(
    all_chunks[:3]
):


    print(
        f"\nChunk {i + 1}"
    )


    print(
        "Reference File:",
        item["source"]
    )


    print(
        "Chunk Number:",
        item["chunk_number"]
    )


    print(
        "Length:",
        len(item["text"])
    )


    print("\nText:")


    print(
        item["text"]
    )


    print(
        "-" * 70
    )


# ============================================================
# 6. Embedding function
# ============================================================

def create_embedding(text):

    vector = model.encode(

        text,

        convert_to_numpy=True

    )


    vector = np.array(

        vector,

        dtype="float32"

    )


    return vector


# ============================================================
# 7. Generate embeddings for all chunks
# ============================================================

print("\n")
print("=" * 70)
print("GENERATING EMBEDDINGS")
print("=" * 70)


embeddings = []


for i, item in enumerate(all_chunks):


    vector = create_embedding(

        item["text"]

    )


    embeddings.append(
        vector
    )


    print(

        f"Embedded chunk "
        f"{i + 1}/"
        f"{len(all_chunks)}"

    )


# Convert list to NumPy matrix
embedding_matrix = np.array(

    embeddings,

    dtype="float32"

)


print(
    "\nEmbedding Matrix Shape:",
    embedding_matrix.shape
)


# ============================================================
# 8. Normalize vectors
# ============================================================

# We want cosine similarity.
#
# FAISS IndexFlatIP calculates inner product.
#
# If vectors are normalized:
#
# Inner Product = Cosine Similarity


faiss.normalize_L2(
    embedding_matrix
)


print(
    "\nEmbeddings normalized."
)


# ============================================================
# 9. Create FAISS Vector Database
# ============================================================

dimension = (
    embedding_matrix.shape[1]
)


index = faiss.IndexFlatIP(
    dimension
)


index.add(
    embedding_matrix
)


print(
    "\nFAISS Vector Database created."
)


print(
    "Number of stored vectors:",
    index.ntotal
)


print(
    "Vector dimension:",
    dimension
)


# ============================================================
# 10. Save FAISS Vector Database
# ============================================================

faiss.write_index(

    index,

    "vector_database.faiss"

)


# ============================================================
# 11. Save metadata
# ============================================================

metadata = {

    "embedding_model":
        EMBEDDING_MODEL,

    "dimension":
        dimension,

    "chunks":
        all_chunks

}


with open(

    "chunk_metadata.pkl",

    "wb"

) as file:

    pickle.dump(

        metadata,

        file

    )


# ============================================================
# 12. Phase 1 completed
# ============================================================

print("\n")
print("=" * 70)
print("PHASE 1 COMPLETED")
print("=" * 70)


print(
    "\nFiles created:"
)


print(
    "1. vector_database.faiss"
)


print(
    "2. chunk_metadata.pkl"
)


print(
    "\nKeep these two files."
)


print(
    "\nYou do NOT need to upload "
    "the original TXT files again "
    "during Phase 2."
)


# ============================================================
# 13. Download database files
# ============================================================

files.download(
    "vector_database.faiss"
)


files.download(
    "chunk_metadata.pkl"
)

PHASE 1 - BUILD VECTOR DATABASE

Upload exactly 3 TXT files from different fields.

Example:
- football.txt
- saudi_food.txt
- artificial_intelligence.txt


KeyboardInterrupt: 

In [13]:
# ============================================================
# CELL 3
# PHASE 2
#
# Load existing Vector DB
# Embed ONLY the query
# Calculate Cosine Similarity
# Return Top 3
# Print Question + Answer + References + Scores
# ============================================================


print("=" * 70)
print("PHASE 2 - VECTOR SIMILARITY SEARCH")
print("=" * 70)


# ============================================================
# 1. Upload existing database files
# ============================================================

print(
    "\nUpload the two files "
    "created in Phase 1:"
)


print(
    "1. vector_database.faiss"
)


print(
    "2. chunk_metadata.pkl"
)


uploaded = files.upload()


# ============================================================
# 2. Check that files exist
# ============================================================

if not os.path.exists(
    "vector_database.faiss"
):

    raise FileNotFoundError(

        "vector_database.faiss "
        "was not uploaded."

    )


if not os.path.exists(
    "chunk_metadata.pkl"
):

    raise FileNotFoundError(

        "chunk_metadata.pkl "
        "was not uploaded."

    )


# ============================================================
# 3. Load existing FAISS Vector Database
# ============================================================

index = faiss.read_index(

    "vector_database.faiss"

)


print(
    "\nExisting FAISS Vector Database loaded."
)


print(
    "Stored vectors:",
    index.ntotal
)


# ============================================================
# 4. Load metadata
# ============================================================

with open(

    "chunk_metadata.pkl",

    "rb"

) as file:

    metadata = pickle.load(
        file
    )


all_chunks = metadata["chunks"]


saved_model_name = (
    metadata["embedding_model"]
)


saved_dimension = (
    metadata["dimension"]
)


print(
    "\nMetadata loaded."
)


print(
    "Stored chunks:",
    len(all_chunks)
)


print(
    "Embedding model used in Phase 1:",
    saved_model_name
)


print(
    "Vector dimension:",
    saved_dimension
)


# ============================================================
# 5. Verify same embedding model
# ============================================================

if saved_model_name != EMBEDDING_MODEL:

    raise ValueError(

        "The embedding model used "
        "in Phase 2 does not match "
        "the model used in Phase 1."

    )


print(
    "\nSame embedding model confirmed."
)


# ============================================================
# 6. Query embedding function
# ============================================================

def embed_query(text):


    vector = model.encode(

        text,

        convert_to_numpy=True

    )


    vector = np.array(

        vector,

        dtype="float32"

    )


    return vector


# ============================================================
# 7. Vector Similarity Search function
# ============================================================

def vector_similarity_search(
    question,
    top_k=3
):


    # --------------------------------------------------------
    # Convert user query into vector
    # --------------------------------------------------------

    query_vector = embed_query(

        question

    )


    # --------------------------------------------------------
    # Convert to 2D array
    # --------------------------------------------------------

    query_vector = np.array(

        [query_vector],

        dtype="float32"

    )


    # --------------------------------------------------------
    # Check vector dimension
    # --------------------------------------------------------

    if query_vector.shape[1] != index.d:

        raise ValueError(

            f"Query vector dimension "
            f"{query_vector.shape[1]} "
            f"does not match FAISS "
            f"dimension {index.d}."

        )


    # --------------------------------------------------------
    # Normalize query vector
    # --------------------------------------------------------

    faiss.normalize_L2(
        query_vector
    )


    # --------------------------------------------------------
    # Make sure Top-K is valid
    # --------------------------------------------------------

    top_k = min(

        top_k,

        index.ntotal

    )


    # --------------------------------------------------------
    # Search FAISS
    # --------------------------------------------------------

    similarity_scores, indices = (

        index.search(

            query_vector,

            top_k

        )

    )


    results = []


    # --------------------------------------------------------
    # Build ranked results
    # --------------------------------------------------------

    for rank in range(top_k):


        chunk_index = (
            indices[0][rank]
        )


        score = (
            similarity_scores[0][rank]
        )


        if chunk_index == -1:

            continue


        chunk = all_chunks[
            chunk_index
        ]


        results.append({

            "rank":
                rank + 1,

            "score":
                float(score),

            "source":
                chunk["source"],

            "chunk_number":
                chunk["chunk_number"],

            "text":
                chunk["text"]

        })


    return results


# ============================================================
# 8. Accept user question
# ============================================================

print("\n")
print("=" * 70)
print("ENTER YOUR QUESTION")
print("=" * 70)


question = input(
    "\nQuestion: "
)


# ============================================================
# 9. Search Top 3 most relevant chunks
# ============================================================

results = vector_similarity_search(

    question,

    top_k=3

)


# ============================================================
# 10. Print the question
# ============================================================

print("\n")
print("=" * 70)
print("QUESTION")
print("=" * 70)


print(
    question
)


# ============================================================
# 11. Print Top 3 chunks
# ============================================================

print("\n")
print("=" * 70)

print(
    "TOP 3 MOST RELEVANT CHUNKS"
)

print("=" * 70)


for result in results:


    print(
        f"\nRank #{result['rank']}"
    )


    print(
        "Reference File:",
        result["source"]
    )


    print(
        "Chunk Number:",
        result["chunk_number"]
    )


    print(
        "Cosine Similarity Score:",
        round(
            result["score"],
            4
        )
    )


    print(
        "\nRetrieved Text:"
    )


    print(
        result["text"]
    )


    print(
        "\n" + "-" * 70
    )


# ============================================================
# 12. Create answer without Generative AI
# ============================================================

# There is no Gemini/OpenAI generation model.
#
# The most relevant retrieved chunk
# becomes the final answer.


if len(results) > 0:

    answer = results[0]["text"]

else:

    answer = (
        "No relevant information "
        "was found in the Vector Database."
    )


# ============================================================
# 13. Print final Question + Answer
# ============================================================

print("\n")
print("=" * 70)

print(
    "FINAL RESULT"
)

print("=" * 70)


print(
    "\nQuestion:"
)


print(
    question
)


print(
    "\nAnswer:"
)


print(
    answer
)


# ============================================================
# 14. Print references
# ============================================================

print(
    "\nReferences:"
)


for result in results:


    print(

        f"- {result['source']} "

        f"(Chunk "
        f"{result['chunk_number']}, "

        f"Cosine Similarity: "
        f"{result['score']:.4f})"

    )


print("\n")
print("=" * 70)

print(
    "Search completed successfully."
)

print("=" * 70)

PHASE 2 - VECTOR SIMILARITY SEARCH

Upload the two files created in Phase 1:
1. vector_database.faiss
2. chunk_metadata.pkl


Saving chunk_metadata (1).pkl to chunk_metadata (1) (1).pkl

Existing FAISS Vector Database loaded.
Stored vectors: 25

Metadata loaded.
Stored chunks: 25
Embedding model used in Phase 1: sentence-transformers/all-MiniLM-L6-v2
Vector dimension: 384

Same embedding model confirmed.


ENTER YOUR QUESTION

Question: What is Manchester City?


QUESTION
What is Manchester City?


TOP 3 MOST RELEVANT CHUNKS

Rank #1
Reference File: Football-Clubs (1).txt
Chunk Number: 10
Cosine Similarity Score: 0.6068

Retrieved Text:
Manchester City has become one of the most dominant football clubs of the modern era. The English club has enjoyed tremendous domestic success, particularly under manager Pep Guardiola. Manchester City is known for possession-based attacking football and has featured stars such as Kevin De Bruyne, Sergio Agüero, David Silva, Erling Haaland, and Vincent Kompany. In 2023, Manchester City achieved a historic treble by winning the Premier League, FA Cup, and UEFA Champions League 